# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/space-0d/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [16]:
!pip install -q duckdb huggingface_hub


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row means one content item for one client on one report date. I use fact_content_daily_performance as the main table, with dim_content for content metadata and dim_clients for client history/availability when needed. I develop on the March 2026 partition. At the decision moment, I want to predict whether a content item will decline in the following period, using only information available before that outcome period. I deliberately exclude future outcome information and identifiers such as client/content IDs from the model features.

The warehouse documentation confirms that fact_content_daily_performance has the grain report_date × client × content and is partitioned by month.

In [20]:
q1 = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").fetchdf()

q1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Put this in the Markdown/text cell

Features

I will use:

gsc_impressions — previous search visibility/traffic volume available before the decision moment.
gsc_clicks — previous search clicks available before the decision moment.
gsc_avg_position — previous search position available before the decision moment.
ga4_sessions — previous analytics sessions, only when ga4_data_available IS TRUE.
ga4_pageviews — previous analytics pageviews, only when ga4_data_available IS TRUE.

These are useful because they describe the content's historical search and analytics performance before the future outcome is measured.

Label / proxy

The label is future content decline, defined from a future comparison of impressions. In the starter data, the official decline label is is_declining_label, where the value is 1 when trend_direction == "down". trend_direction is based on the change between the last 30 days and previous 30 days.

Context

client_hash_id, content_hash_id, and report_date are context fields. They are useful for grouping, joining, filtering and splitting, but they are not model features. The data rules explicitly say IDs are for grouping/joining/splitting only.

Excluded

I exclude trend_pct and trend_direction because they are derived from the outcome and therefore leak the label. I also exclude client_hash_id and content_hash_id from model features because they are identifiers. I do not treat zero-filled GA4 values as genuine zero engagement when ga4_data_available is not true.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
q1 = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").fetchdf()

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [23]:
q2 = con.execute("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
""").fetchdf()

q2

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [24]:
q3 = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
);
""").fetchdf()

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,gsc_available_rows
0,9841378,413966,3611061


I selected five features that are knowable before the prediction moment:

gsc_impressions — historical Google Search Console impressions.
gsc_clicks — historical Google Search Console clicks.
gsc_avg_position — historical average search position.
ga4_sessions — historical GA4 sessions, when ga4_data_available IS TRUE.
ga4_pageviews — historical GA4 pageviews, when ga4_data_available IS TRUE.

I exclude identifiers such as client_hash_id and content_hash_id from the feature set. I also exclude trend_pct and trend_direction because they are used to derive the decline label and would cause leakage.

These features are available only from information observed before the prediction/outcome window. GA4 features are used only when ga4_data_available IS TRUE; NULL or FALSE availability is not interpreted as genuine zero engagement.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: The warehouse is an unbalanced panel, meaning different clients have different amounts of historical data available. Some clients have much longer history than others, so the same calendar window does not provide equal historical coverage for every client. Therefore, per-client history coverage should be checked before defining time windows.

GA4 limitation: GA4 availability is also incomplete. Before a client's ga4_data_start, GA4 values may be zero-filled with ga4_data_available = FALSE, and the availability flag can also be NULL. Therefore, missing or unavailable GA4 data should not be interpreted as zero engagement; filtering should use ga4_data_available IS TRUE.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.